# Fase 3: Preparacion, limpieza y modelado de datos

Este notebook prepara la base limpia y crea las tablas que luego se usan en el analisis y en el dashboard.

In [1]:
# Importamos las librerias necesarias para limpiar y resumir los datos.
from pathlib import Path
import pandas as pd
import numpy as np

# Definimos las rutas principales del proyecto.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Creamos carpetas si todavia no existen.
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carga del dataset original

In [2]:
# Cargamos el archivo original tal como viene desde la fuente.
raw_path = DATA_DIR / "raw_hr_analytics.csv"
df_raw = pd.read_csv(raw_path)

# Mostramos el tamano general del dataset.
print(f"Filas: {df_raw.shape[0]:,}")
print(f"Columnas: {df_raw.shape[1]}")

Filas: 14,999
Columnas: 10


In [3]:
# Revisamos algunas filas y los tipos de dato iniciales.
display(df_raw.head())
display(df_raw.dtypes.to_frame("tipo_dato"))

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


,tipo_dato
satisfaction_level,float64
last_evaluation,float64
number_project,int64
average_montly_hours,int64
time_spend_company,int64
Work_accident,int64
left,int64
promotion_last_5years,int64
Department,str
salary,str


## 2. Auditoria de calidad

In [4]:
# Contamos valores faltantes por columna para saber si hay vacios.
missing = df_raw.isna().sum().to_frame("valores_faltantes")
missing["porcentaje"] = (missing["valores_faltantes"] / len(df_raw) * 100).round(2)
display(missing)

,valores_faltantes,porcentaje
satisfaction_level,0,0.0
last_evaluation,0,0.0
number_project,0,0.0
average_montly_hours,0,0.0
time_spend_company,0,0.0
Work_accident,0,0.0
left,0,0.0
promotion_last_5years,0,0.0
Department,0,0.0
salary,0,0.0


In [5]:
# Revisamos registros duplicados exactos.
duplicated_rows_after_first = int(df_raw.duplicated().sum())
duplicated_rows_in_groups = int(df_raw.duplicated(keep=False).sum())

print("Duplicados posteriores al primer registro:", duplicated_rows_after_first)
print("Filas involucradas en grupos duplicados:", duplicated_rows_in_groups)

# Mostramos una pequena muestra de los duplicados para inspeccion manual.
display(df_raw[df_raw.duplicated(keep=False)].head(10))

Duplicados posteriores al primer registro: 3008
Filas involucradas en grupos duplicados: 5346


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low
5,0.41,0.50,2,153,3,0,1,0,sales,low
6,0.10,0.77,6,247,4,0,1,0,sales,low
7,0.92,0.85,5,259,5,0,1,0,sales,low
8,0.89,1.00,5,224,5,0,1,0,sales,low
9,0.42,0.53,2,142,3,0,1,0,sales,low


## 3. Limpieza y transformacion

In [6]:
# Hacemos una copia de trabajo y corregimos nombres de columnas.
df = df_raw.copy()
df.columns = [col.strip() for col in df.columns]
df = df.rename(
    columns={
        "average_montly_hours": "average_monthly_hours",
        "Work_accident": "work_accident",
        "Department": "department",
    }
)

# Verificamos los nombres corregidos.
print(df.columns.tolist())

['satisfaction_level', 'last_evaluation', 'number_project', 'average_monthly_hours', 'time_spend_company', 'work_accident', 'left', 'promotion_last_5years', 'department', 'salary']


In [7]:
# Calculamos medidas base que luego usaremos para crear categorias.
missing_total = int(df.isna().sum().sum())
satisfaction_median = float(df["satisfaction_level"].median())
hours_q1 = float(df["average_monthly_hours"].quantile(0.25))
hours_q3 = float(df["average_monthly_hours"].quantile(0.75))

# Definimos traducciones y ordenes utiles para el dashboard.
salary_order = {"low": 1, "medium": 2, "high": 3}
salary_label = {"low": "Bajo", "medium": "Medio", "high": "Alto"}
department_label = {
    "IT": "IT",
    "RandD": "I+D",
    "accounting": "Contabilidad",
    "hr": "Recursos Humanos",
    "management": "Gerencia",
    "marketing": "Marketing",
    "product_mng": "Producto",
    "sales": "Ventas",
    "support": "Soporte",
    "technical": "Tecnico",
}

print("Mediana de satisfaccion:", round(satisfaction_median, 4))
print("Q1 de horas:", round(hours_q1, 2))
print("Q3 de horas:", round(hours_q3, 2))

Mediana de satisfaccion: 0.64
Q1 de horas: 156.0
Q3 de horas: 245.0


In [8]:
# Creamos columnas base que usaremos en reportes y visualizaciones.
df["duplicate_exact_record"] = df.duplicated(keep=False).astype(int)
df["employee_count"] = 1
df["left_label"] = np.where(df["left"].eq(1), "Se va", "Permanece")
df["salary_order"] = df["salary"].map(salary_order)
df["salary_label"] = df["salary"].map(salary_label)
df["department_label"] = df["department"].map(department_label).fillna(df["department"])

display(df[["department", "department_label", "salary", "salary_label", "left", "left_label"]].head())

,department,department_label,salary,salary_label,left,left_label
0,sales,Ventas,low,Bajo,1,Se va
1,sales,Ventas,medium,Medio,1,Se va
2,sales,Ventas,medium,Medio,1,Se va
3,sales,Ventas,low,Bajo,1,Se va
4,sales,Ventas,low,Bajo,1,Se va


In [9]:
# Convertimos variables continuas o binarias en categorias faciles de leer.
df["satisfaction_category"] = np.where(df["satisfaction_level"].lt(satisfaction_median), "Baja", "Alta")
df["promotion_label"] = np.where(df["promotion_last_5years"].eq(1), "Con promocion", "Sin promocion")
df["work_accident_label"] = np.where(df["work_accident"].eq(1), "Con accidente", "Sin accidente")
df["workload_category"] = np.select(
    [
        df["average_monthly_hours"].lt(hours_q1),
        df["average_monthly_hours"].gt(hours_q3),
    ],
    ["Baja carga", "Alta carga"],
    default="Carga media",
)
df["tenure_category"] = pd.cut(
    df["time_spend_company"],
    bins=[1, 2, 4, 10],
    labels=["2 anios", "3-4 anios", "5+ anios"],
    include_lowest=True,
).astype(str)

display(
    df[
        [
            "satisfaction_level",
            "satisfaction_category",
            "average_monthly_hours",
            "workload_category",
            "time_spend_company",
            "tenure_category",
        ]
    ].head()
)

,satisfaction_level,satisfaction_category,average_monthly_hours,workload_category,time_spend_company,tenure_category
0,0.38,Baja,157,Carga media,3,3-4 anios
1,0.80,Alta,262,Alta carga,6,5+ anios
2,0.11,Baja,272,Alta carga,4,3-4 anios
3,0.72,Alta,223,Carga media,5,5+ anios
4,0.37,Baja,159,Carga media,3,3-4 anios


In [10]:
# Guardamos la base limpia y un resumen corto de auditoria.
df.to_csv(DATA_DIR / "clean_hr_analytics.csv", index=False, encoding="utf-8")

audit = {
    "raw_rows": int(len(df_raw)),
    "clean_rows": int(len(df)),
    "columns_raw": int(df_raw.shape[1]),
    "columns_clean": int(df.shape[1]),
    "missing_total": missing_total,
    "duplicated_rows_detected": duplicated_rows_after_first,
    "duplicated_rows_after_first": duplicated_rows_after_first,
    "duplicated_rows_in_groups": duplicated_rows_in_groups,
    "duplicated_rows_removed": 0,
    "satisfaction_median": round(satisfaction_median, 4),
    "hours_q1": round(hours_q1, 2),
    "hours_q3": round(hours_q3, 2),
}

pd.DataFrame([audit]).to_csv(REPORTS_DIR / "auditoria_limpieza.csv", index=False)
display(pd.DataFrame([audit]))

,raw_rows,clean_rows,columns_raw,columns_clean,missing_total,duplicated_rows_detected,duplicated_rows_after_first,duplicated_rows_in_groups,duplicated_rows_removed,satisfaction_median,hours_q1,hours_q3
0,14999,14999,10,21,0,3008,3008,5346,0,0.64,156.0,245.0


## 4. Dimensiones y metricas para BI

In [11]:
# Resumimos la informacion por departamento.
department = (
    df.groupby(["department", "department_label"], as_index=False)
    .agg(
        employees=("employee_count", "sum"),
        left_count=("left", "sum"),
        turnover_rate=("left", "mean"),
        avg_satisfaction=("satisfaction_level", "mean"),
        avg_monthly_hours=("average_monthly_hours", "mean"),
        avg_projects=("number_project", "mean"),
        promotion_rate=("promotion_last_5years", "mean"),
        accident_rate=("work_accident", "mean"),
    )
    .sort_values("turnover_rate", ascending=False)
)
department["turnover_rate_pct"] = department["turnover_rate"].mul(100).round(2)
department["avg_satisfaction"] = department["avg_satisfaction"].round(3)
department["avg_monthly_hours"] = department["avg_monthly_hours"].round(1)
department["avg_projects"] = department["avg_projects"].round(2)
department["promotion_rate_pct"] = department["promotion_rate"].mul(100).round(2)
department["accident_rate_pct"] = department["accident_rate"].mul(100).round(2)

display(department.head())

,department,department_label,employees,left_count,turnover_rate,avg_satisfaction,avg_monthly_hours,avg_projects,promotion_rate,accident_rate,turnover_rate_pct,promotion_rate_pct,accident_rate_pct
3,hr,Recursos Humanos,739,215,0.290934,0.599,198.7,3.65,0.020298,0.120433,29.09,2.03,12.04
2,accounting,Contabilidad,767,204,0.265971,0.582,201.2,3.83,0.018253,0.125163,26.60,1.83,12.52
9,technical,Tecnico,2720,697,0.256250,0.608,202.5,3.88,0.010294,0.140074,25.62,1.03,14.01
8,support,Soporte,2229,555,0.248991,0.618,200.8,3.80,0.008973,0.154778,24.90,0.90,15.48
7,sales,Ventas,4140,1014,0.244928,0.614,200.9,3.78,0.024155,0.141787,24.49,2.42,14.18


In [12]:
# Resumimos la informacion por nivel salarial.
salary = (
    df.groupby(["salary", "salary_label", "salary_order"], as_index=False)
    .agg(
        employees=("employee_count", "sum"),
        left_count=("left", "sum"),
        turnover_rate=("left", "mean"),
        avg_satisfaction=("satisfaction_level", "mean"),
        avg_monthly_hours=("average_monthly_hours", "mean"),
    )
    .sort_values("salary_order")
)
salary["turnover_rate_pct"] = salary["turnover_rate"].mul(100).round(2)
salary["avg_satisfaction"] = salary["avg_satisfaction"].round(3)
salary["avg_monthly_hours"] = salary["avg_monthly_hours"].round(1)

display(salary)

,salary,salary_label,salary_order,employees,left_count,turnover_rate,avg_satisfaction,avg_monthly_hours,turnover_rate_pct
1,low,Bajo,1,7316,2172,0.296884,0.601,201.0,29.69
2,medium,Medio,2,6446,1317,0.204313,0.622,201.3,20.43
0,high,Alto,3,1237,82,0.066289,0.637,199.9,6.63


In [13]:
# Resumimos la informacion por categoria de satisfaccion.
satisfaction = (
    df.groupby("satisfaction_category", as_index=False)
    .agg(
        employees=("employee_count", "sum"),
        left_count=("left", "sum"),
        turnover_rate=("left", "mean"),
        avg_satisfaction=("satisfaction_level", "mean"),
        avg_monthly_hours=("average_monthly_hours", "mean"),
    )
    .sort_values("satisfaction_category")
)
satisfaction["turnover_rate_pct"] = satisfaction["turnover_rate"].mul(100).round(2)
satisfaction["avg_satisfaction"] = satisfaction["avg_satisfaction"].round(3)
satisfaction["avg_monthly_hours"] = satisfaction["avg_monthly_hours"].round(1)

display(satisfaction)

,satisfaction_category,employees,left_count,turnover_rate,avg_satisfaction,avg_monthly_hours,turnover_rate_pct
0,Alta,7665,959,0.125114,0.815,206.6,12.51
1,Baja,7334,2612,0.356149,0.402,195.2,35.61


In [14]:
# Resumimos la informacion por categoria de carga laboral.
workload = (
    df.groupby("workload_category", as_index=False)
    .agg(
        employees=("employee_count", "sum"),
        left_count=("left", "sum"),
        turnover_rate=("left", "mean"),
        avg_satisfaction=("satisfaction_level", "mean"),
        avg_monthly_hours=("average_monthly_hours", "mean"),
    )
    .sort_values("avg_monthly_hours")
)
workload["turnover_rate_pct"] = workload["turnover_rate"].mul(100).round(2)
workload["avg_satisfaction"] = workload["avg_satisfaction"].round(3)
workload["avg_monthly_hours"] = workload["avg_monthly_hours"].round(1)

display(workload)

,workload_category,employees,left_count,turnover_rate,avg_satisfaction,avg_monthly_hours,turnover_rate_pct
1,Baja carga,3680,1326,0.360326,0.558,138.0,36.03
2,Carga media,7628,868,0.113791,0.667,200.1,11.38
0,Alta carga,3691,1377,0.373070,0.555,266.0,37.31


In [15]:
# Resumimos la informacion por antiguedad en la empresa.
tenure = (
    df.groupby("tenure_category", as_index=False)
    .agg(
        employees=("employee_count", "sum"),
        left_count=("left", "sum"),
        turnover_rate=("left", "mean"),
        avg_satisfaction=("satisfaction_level", "mean"),
        avg_monthly_hours=("average_monthly_hours", "mean"),
    )
)
tenure["tenure_category"] = pd.Categorical(
    tenure["tenure_category"],
    categories=["2 anios", "3-4 anios", "5+ anios"],
    ordered=True,
)
tenure = tenure.sort_values("tenure_category")
tenure["turnover_rate_pct"] = tenure["turnover_rate"].mul(100).round(2)
tenure["avg_satisfaction"] = tenure["avg_satisfaction"].round(3)
tenure["avg_monthly_hours"] = tenure["avg_monthly_hours"].round(1)
tenure["tenure_category"] = tenure["tenure_category"].astype(str)

display(tenure)

,tenure_category,employees,left_count,turnover_rate,avg_satisfaction,avg_monthly_hours,turnover_rate_pct
0,2 anios,3244,53,0.016338,0.697,200.1,1.63
1,3-4 anios,9000,2476,0.275111,0.581,197.1,27.51
2,5+ anios,2755,1042,0.378221,0.617,215.1,37.82


In [16]:
# Creamos una tabla combinada por departamento, salario y satisfaccion.
combo = (
    df.groupby(["department_label", "salary_label", "satisfaction_category"], as_index=False)
    .agg(
        employees=("employee_count", "sum"),
        left_count=("left", "sum"),
        avg_satisfaction=("satisfaction_level", "mean"),
        avg_monthly_hours=("average_monthly_hours", "mean"),
        avg_projects=("number_project", "mean"),
    )
    .sort_values(["department_label", "salary_label", "satisfaction_category"])
)
combo["turnover_rate"] = combo["left_count"] / combo["employees"]
combo["turnover_rate_pct"] = combo["turnover_rate"].mul(100).round(2)
combo["avg_satisfaction"] = combo["avg_satisfaction"].round(3)
combo["avg_monthly_hours"] = combo["avg_monthly_hours"].round(1)
combo["avg_projects"] = combo["avg_projects"].round(2)

display(combo.head())

,department_label,salary_label,satisfaction_category,employees,left_count,avg_satisfaction,avg_monthly_hours,avg_projects,turnover_rate,turnover_rate_pct
0,Contabilidad,Alto,Alta,35,0,0.821,203.6,4.03,0.000000,0.00
1,Contabilidad,Alto,Baja,39,5,0.428,208.0,3.79,0.128205,12.82
2,Contabilidad,Bajo,Alta,152,18,0.817,204.8,3.93,0.118421,11.84
3,Contabilidad,Bajo,Baja,206,81,0.395,196.3,3.71,0.393204,39.32
4,Contabilidad,Medio,Alta,167,23,0.807,202.9,3.84,0.137725,13.77


In [17]:
# Exportamos todas las tablas resumidas a archivos CSV.
aggregates = {
    "department": department,
    "salary": salary,
    "satisfaction": satisfaction,
    "workload": workload,
    "tenure": tenure,
    "dashboard_combo": combo,
}

for name, frame in aggregates.items():
    frame.to_csv(DATA_DIR / f"{name}_metrics.csv", index=False, encoding="utf-8")

print("Archivos agregados exportados correctamente.")

Archivos agregados exportados correctamente.


## 5. Lectura inicial del EDA

In [18]:
# Mostramos algunos resultados rapidos para interpretar la base limpia.
print("Tasa de rotacion general:", round(df["left"].mean() * 100, 2), "%")
print("Satisfaccion promedio:", round(df["satisfaction_level"].mean(), 3))
print("Horas mensuales promedio:", round(df["average_monthly_hours"].mean(), 1))
print("Departamento con mayor rotacion:", department.iloc[0]["department_label"])
print("Salario con mayor rotacion:", salary.sort_values("turnover_rate", ascending=False).iloc[0]["salary_label"])

Tasa de rotacion general: 23.81 %
Satisfaccion promedio: 0.613
Horas mensuales promedio: 201.1
Departamento con mayor rotacion: Recursos Humanos
Salario con mayor rotacion: Bajo


## 6. Archivos generados

In [19]:
# Listamos los archivos principales que deja este notebook.
generated_files = [
    DATA_DIR / "clean_hr_analytics.csv",
    DATA_DIR / "department_metrics.csv",
    DATA_DIR / "salary_metrics.csv",
    DATA_DIR / "satisfaction_metrics.csv",
    DATA_DIR / "workload_metrics.csv",
    DATA_DIR / "tenure_metrics.csv",
    DATA_DIR / "dashboard_combo_metrics.csv",
    REPORTS_DIR / "auditoria_limpieza.csv",
]

for path in generated_files:
    print(path)

d:\Fase 3 Analisis de datos\Data-Analysis-3\data\clean_hr_analytics.csv
d:\Fase 3 Analisis de datos\Data-Analysis-3\data\department_metrics.csv
d:\Fase 3 Analisis de datos\Data-Analysis-3\data\salary_metrics.csv
d:\Fase 3 Analisis de datos\Data-Analysis-3\data\satisfaction_metrics.csv
d:\Fase 3 Analisis de datos\Data-Analysis-3\data\workload_metrics.csv
d:\Fase 3 Analisis de datos\Data-Analysis-3\data\tenure_metrics.csv
d:\Fase 3 Analisis de datos\Data-Analysis-3\data\dashboard_combo_metrics.csv
d:\Fase 3 Analisis de datos\Data-Analysis-3\reports\auditoria_limpieza.csv
